In [37]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("covid-combo.db")

def sql(query: str, **kwargs) -> pd.DataFrame:
    """Run a query against covid-combo.db and return a DataFrame."""
    return pd.read_sql_query(query, conn, **kwargs)

In [38]:
vbm_locations = sql("SELECT COUNT(DISTINCT location) FROM vaccinations_by_manufacturer")
covid_locations = sql("SELECT COUNT(DISTINCT location) FROM covid_data")
print(f"Problem limited to {vbm_locations.iloc[0, 0]} locations in vaccinations_by_manufacturer and {covid_locations.iloc[0, 0]} locations in covid_data")
print(f"vaccinations_by_manufacturer: {vbm_locations.iloc[0, 0]} distinct locations")  
print(f"covid_data:                   {covid_locations.iloc[0, 0]} distinct locations")  
matching_locations = sql("""
    SELECT COUNT(*)
    FROM (SELECT DISTINCT location FROM vaccinations_by_manufacturer) AS v
    INNER JOIN (SELECT DISTINCT location FROM covid_data) AS c
        ON v.location = c.location
""")
print(f"matching in both:             {matching_locations.iloc[0, 0]} locations")

Problem limited to 44 locations in vaccinations_by_manufacturer and 255 locations in covid_data
vaccinations_by_manufacturer: 44 distinct locations
covid_data:                   255 distinct locations
matching in both:             43 locations


In [39]:
excess_mort = sql("""
    SELECT COUNT(excess_mortality_cumulative_per_million) AS non_null,
           COUNT(*)                                       AS total
    FROM covid_data
""")
non_null, total = excess_mort.iloc[0]
print(f"excess_mortality_cumulative_per_million: {non_null} of {total} rows have a value")

excess_mortality_cumulative_per_million: 13411 of 429435 rows have a value


In [40]:
# gdp_annual: source from the World Bank API
# in covid_data but not in gdp_annual
covid_only = sql("""
    SELECT DISTINCT iso_code, location FROM covid_data
    WHERE iso_code NOT LIKE 'OWID_%' AND  iso_code NOT IN (SELECT iso_code FROM gdp_annual WHERE iso_code IS NOT NULL)
    ORDER BY location
""")

print(f"\nOnly in covid_data: {len(covid_only)}")
print(covid_only.to_string(index=False))


Only in covid_data: 22
iso_code                        location
     AIA                        Anguilla
     BES Bonaire Sint Eustatius and Saba
     COK                    Cook Islands
     FLK                Falkland Islands
     GUF                   French Guiana
     GLP                      Guadeloupe
     GGY                        Guernsey
     JEY                          Jersey
     MTQ                      Martinique
     MYT                         Mayotte
     MSR                      Montserrat
     NIU                            Niue
     PCN                        Pitcairn
     REU                         Reunion
     BLM                Saint Barthelemy
     SHN                    Saint Helena
     SPM       Saint Pierre and Miquelon
     TWN                          Taiwan
     TKL                         Tokelau
     VAT                         Vatican
     WLF               Wallis and Futuna
     ESH                  Western Sahara


In [41]:
query = """SELECT f.year,
       f.forecast_real_gdp_growth_pct AS expected,
       a.gdp_growth_pct              AS actual
FROM imf_weo_2019_forecast f
JOIN gdp_annual a ON a.iso_code = f.iso_code AND a.year = f.year
WHERE f.iso_code = 'BEL' AND f.value_type = 'forecast';"""

sql(query)


,year,expected,actual
0,2020,1.300,-4.792984
1,2021,1.305,6.254457
2,2022,1.319,4.024609
3,2023,1.359,1.561255
4,2024,1.362,1.097754
